# Подготовка данных для восьми горизонтов
Запустить все ячейки сверху вниз. Один прогноз в день из 10:30 для каждого маршрута. Для горизонта h время цели равно 10:30 + h × 30 минут. После подготовки признаков ноутбук обучает модели и собирает submission.csv. Для запуска выберите GPU в настройках Kaggle.


In [2]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        print(p)

/kaggle/input/datasets/sptrent/rwb-hack/test_solo_track.parquet
/kaggle/input/datasets/sptrent/rwb-hack/train_solo_track.parquet


In [3]:
import numpy as np
import pandas as pd

DATA = Path("/kaggle/input/datasets/sptrent/rwb-hack/")
TRAIN_FILE = "train_solo_track.parquet"
TEST_FILE = "test_solo_track.parquet"
if not (DATA / TRAIN_FILE).is_file() and Path("/kaggle/input").exists():
    matches = [p.parent for p in Path("/kaggle/input").rglob(TRAIN_FILE)
               if (p.parent / TEST_FILE).is_file()]
    if len(matches) != 1:
        raise ValueError(f"Укажите DATA вручную. Найденные папки: {matches}")
    DATA = matches[0]
OUT = Path("artifacts/features")
OUT.mkdir(parents=True, exist_ok=True)
TARGET = "target_1h"
STATUSES = [f"status_{i}" for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)

## Получасовая сетка
Проверяем полноту панели перед использованием позиционных лагов. Строки массивов - моменты времени, столбцы - маршруты.


In [4]:
history = pd.read_parquet(DATA / TRAIN_FILE).sort_values(["timestamp", "route_id"])
test = pd.read_parquet(DATA / TEST_FILE)
times = pd.DatetimeIndex(history.timestamp.unique())
routes = np.sort(history.route_id.unique())
if not (history.notna().all().all() and test.notna().all().all()):
    raise ValueError("В данных есть пропуски")
if not (not history.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторяется ключ маршрута и времени")
if not (times.equals(pd.date_range(times[0], times[-1], freq=STEP))):
    raise ValueError("Нарушена получасовая сетка")
if not (len(history) == len(times) * len(routes)):
    raise ValueError("Неполная панель маршрутов")
if not (test.id.is_unique and not test.duplicated(["route_id", "timestamp"]).any()):
    raise ValueError("Повторы идентификаторов или ключей теста")
end = times[-1]
if not ((end.hour, end.minute) == (10, 30)):
    raise ValueError("Последняя точка истории должна быть в 10:30")
expected = pd.MultiIndex.from_product([routes, [end + h*STEP for h in range(1, 9)]],
                                      names=["route_id", "timestamp"])
if not (expected.difference(pd.MultiIndex.from_frame(test[["route_id", "timestamp"]])).empty):
    raise ValueError("Тест не покрывает все ожидаемые маршруты и горизонты")
if not (len(test) == len(expected)):
    raise ValueError("Число строк теста отличается от ожидаемого")
values = {name: history[name].to_numpy().reshape(len(times), len(routes))
          for name in [TARGET, *STATUSES]}
del history

## Общие признаки по истории до 10:30
Для всех маршрутов граница истории одинакова. Начинаем после четырёх полных недель, чтобы все сезонные средние имели одинаковое число наблюдений.


In [5]:
origins = np.flatnonzero((times.hour == 10) & (times.minute == 30))
origins = origins[origins >= 28 * 48]
if not (origins.size and origins[-1] == len(times) - 1):
    raise ValueError("Недостаточно истории или отсутствует последняя точка старта")

def window_mean(array, positions, steps):
    return sum(array[positions - k].astype(float) for k in range(steps)) / steps

target = values[TARGET]
common = {}
for k in (0, 1, 2, 4):
    common[f"target_back_{k}"] = target[origins - k]
for name, array in values.items():
    for steps in (4, 12):
        common[f"{name}_mean_{steps}"] = window_mean(array, origins, steps)
    if name in STATUSES:
        common[f"{name}_latest"] = array[origins]
        common[f"{name}_change_1h"] = array[origins] - array[origins - 2]
common["target_vs_mean_6h"] = target[origins] - common[f"{TARGET}_mean_12"]
morning = common[f"{TARGET}_mean_4"]
usual = sum(window_mean(target, origins - day*48, 4) for day in range(1, 8)) / 7
common["morning_level_ratio"] = np.divide(morning, usual, out=np.full_like(morning, np.nan),
                                           where=usual != 0)

## Признаки слота и сохранение
Каждая строка соответствует одному маршруту, дате и горизонту. Метка y доступна только для истории. Колонки id, cutoff, timestamp и y служебные.


In [6]:
level_by_day = {}
for day in range(29):
    positions = origins - day * 48
    available = positions >= 47
    levels = np.full((len(origins), len(routes)), np.nan)
    levels[available] = window_mean(target, positions[available], 48)
    level_by_day[day] = levels

def horizon_dataset(h):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({
        "cutoff": times[origins].repeat(len(routes)),
        "timestamp": dates.repeat(len(routes)),
        "route_id": np.tile(routes, len(origins)),
    })
    for name, matrix in common.items():
        frame[name] = matrix.ravel()
    for name, index, period in [
        ("weekday", dates.dayofweek, 7),
        ("half_hour", dates.hour * 2 + dates.minute // 30 + 1, 48),
    ]:
        phase = index if name == "weekday" else index - 1
        angle = 2 * np.pi * phase / period
        frame[name + "_sin"] = np.repeat(np.sin(angle), len(routes))
        frame[name + "_cos"] = np.repeat(np.cos(angle), len(routes))
    for days in (1, 2, 7):
        frame[f"slot_days_back_{days}"] = target[slot - days*48].ravel()
    for name, days in [("slot_daily7", range(1, 8)), ("slot_weekly4", (7, 14, 21, 28))]:
        frame[name] = (sum(target[slot - d*48] for d in days) / len(days)).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = level_by_day[day]
        adjusted = target[slot - day * 48] * level_by_day[0] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame["profile28"] = (profile / counts).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = target[origins - 2].copy()
        for position in (origins - 1, origins):
            total = target[position]
            lower, upper = np.maximum(0, total - upper), np.minimum(total, total - lower)
        seasonal = frame[["slot_daily7", "slot_weekly4"]].mean(axis=1).to_numpy()
        frame["overlap"] = np.where((lower <= upper).ravel(),
                                    (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame["overlap"] = np.nan
    future = frame.cutoff.eq(end)
    labeled = frame.loc[~future].copy()
    labeled["y"] = target[slot[:-1]].ravel()
    requests = test.loc[test.timestamp.eq(end + h*STEP)]
    predictions = requests.merge(frame.loc[future], how="left", on=["route_id", "timestamp"],
                                 validate="one_to_one")
    if not (predictions.id.equals(requests.id.reset_index(drop=True))):
        raise ValueError("Изменён порядок идентификаторов")
    if not (predictions.cutoff.eq(end).all()):
        raise ValueError("Для тестовой строки отсутствует история")
    return labeled, predictions

for h in range(1, 9):
    labeled, future = horizon_dataset(h)
    if not (labeled.timestamp.lt(future.timestamp.min()).all()):
        raise ValueError("История пересекается с тестовым периодом")
    if not (labeled.drop(columns=["y"]).columns.difference(future.columns).empty):
        raise ValueError("Наборы признаков не совпадают")
    labeled.to_parquet(OUT / f"history_h{h}.parquet", index=False)
    future.to_parquet(OUT / f"submission_h{h}.parquet", index=False)
    print(f"h={h}: история {len(labeled):,}, тест {len(future):,}")
    del labeled, future

del levels, level_by_day, values, target, array, common, morning, usual

h=1: история 68,000, тест 1,000
h=2: история 68,000, тест 1,000
h=3: история 68,000, тест 1,000
h=4: история 68,000, тест 1,000
h=5: история 68,000, тест 1,000
h=6: история 68,000, тест 1,000
h=7: история 68,000, тест 1,000
h=8: история 68,000, тест 1,000


## Обучение CatBoost на GPU
Последние 14 размеченных дат оставляем для оценки. Предыдущие 14 дат используем для выбора числа деревьев по MAE. Все горизонты используют одинаковые даты. После оценки обучаем каждую модель на всей истории с выбранным числом деревьев.


In [7]:
from catboost import CatBoostRegressor
from catboost.utils import get_gpu_device_count

if get_gpu_device_count() < 1:
    raise RuntimeError("Для обучения включите GPU в настройках Kaggle Notebook.")
MODEL_DIR = OUT.parent / "models"
MODEL_DIR.mkdir(exist_ok=True)
dates = pd.DatetimeIndex(pd.read_parquet(OUT / "history_h1.parquet", columns=["cutoff"]).cutoff.unique()).sort_values()
if len(dates) <= 28:
    raise ValueError("Недостаточно дат для временного разбиения")
tune_start, holdout_start = dates[-28], dates[-14]
print("Начало подбора:", tune_start, "Начало отложенной проверки:", holdout_start)

def make_model(trees):
    return CatBoostRegressor(iterations=trees, depth=7, learning_rate=0.05,
                             loss_function="MAE", eval_metric="MAE", task_type="GPU",
                             devices="0", cat_features=["route_id"], random_seed=42,
                             allow_writing_files=False, verbose=100)

def mean_daily_score(frame, prediction):
    errors = np.asarray(prediction) - frame.y.to_numpy()
    totals = pd.DataFrame({"cutoff": frame.cutoff, "volume": frame.y,
                           "error": errors, "absolute": np.abs(errors)}).groupby("cutoff").sum()
    return ((totals.absolute + totals.error.abs()) / totals.volume).mean()

def score_row(frame, predictions, split):
    errors = np.asarray(predictions) - frame.y.to_numpy()
    volume = frame.y.sum()
    wape = np.abs(errors).sum() / volume
    bias = np.abs(errors.sum()) / volume
    return {"выборка": split, "даты": frame.cutoff.nunique(), "строки": len(frame),
            "WAPE": wape, "|Relative Bias|": bias, "метрика конкурса": wape + bias,
            "среднее по дням": mean_daily_score(frame, predictions)}

def variants(h):
    return [("catboost", 1), ("scaled", 100_000)] if h == 2 else [("catboost", 1)]

def model_columns(data):
    return data.columns.drop(["timestamp", "cutoff", "y", "profile28", "overlap"]).tolist()

split_dates = {"train": dates[dates < tune_start],
               "подбор": dates[(dates >= tune_start) & (dates < holdout_start)],
               "test": dates[dates >= holdout_start]}
display(pd.DataFrame([{"выборка": name, "с": period.min(), "по": period.max(), "даты": len(period)}
                      for name, period in split_dates.items()]))

Начало подбора: 2025-10-04 10:30:00 Начало отложенной проверки: 2025-10-18 10:30:00


,выборка,с,по,даты
0,train,2025-08-25 10:30:00,2025-10-03 10:30:00,40
1,подбор,2025-10-04 10:30:00,2025-10-17 10:30:00,14
2,test,2025-10-18 10:30:00,2025-10-31 10:30:00,14


In [8]:
validation_parts = []
tree_counts = {}
evaluation_tables = {}
for h in range(1, 9):
    data = pd.read_parquet(OUT / f"history_h{h}.parquet")
    columns = model_columns(data)
    train = data.cutoff < tune_start
    tune = (data.cutoff >= tune_start) & (data.cutoff < holdout_start)
    evaluate = data.cutoff >= tune_start
    evaluation = data.loc[evaluate, ["cutoff", "timestamp", "route_id", "y", "profile28", "overlap"]].copy()
    evaluation["horizon"] = h
    evaluation["seasonal"] = data.loc[evaluate, ["slot_daily7", "slot_weekly4"]].mean(axis=1)
    for variant, scale in variants(h):
        model = make_model(1500)
        model.fit(data.loc[train, columns], data.loc[train, "y"] / scale,
                  eval_set=(data.loc[tune, columns], data.loc[tune, "y"] / scale),
                  early_stopping_rounds=100, use_best_model=True)
        tree_counts[h, variant] = model.tree_count_
        evaluation[variant] = (model.predict(data.loc[evaluate, columns]) * scale).clip(min=0)
        test_rows = evaluation.loc[evaluation.cutoff >= holdout_start]
        train_pred = (model.predict(data.loc[train, columns]) * scale).clip(min=0)
        table = pd.DataFrame([score_row(data.loc[train], train_pred, "train"),
                              score_row(test_rows, test_rows[variant], "test")])
        evaluation_tables[h, variant] = table
        print(f"h={h}, {variant}, деревьев: {model.tree_count_}")
        display(table)
        del model
    validation_parts.append(evaluation)
validation = pd.concat(validation_parts, ignore_index=True)
validation.to_parquet(OUT.parent / "validation_predictions.parquet", index=False)

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 135297.7920000	test: 142479.2045714	best: 142479.2045714 (0)	total: 166ms	remaining: 4m 9s
100:	learn: 135296.9344000	test: 142478.3085714	best: 142478.3085714 (99)	total: 3.12s	remaining: 43.3s
200:	learn: 135295.9872000	test: 142477.3942857	best: 142477.3942857 (199)	total: 6.01s	remaining: 38.9s
300:	learn: 135295.1040000	test: 142476.4800000	best: 142476.4800000 (300)	total: 8.92s	remaining: 35.5s
400:	learn: 135294.2464000	test: 142475.5840000	best: 142475.5840000 (400)	total: 11.8s	remaining: 32.4s
500:	learn: 135293.3248000	test: 142474.6971429	best: 142474.6971429 (500)	total: 14.8s	remaining: 29.5s
600:	learn: 135292.4288000	test: 142473.7828571	best: 142473.7828571 (600)	total: 17.7s	remaining: 26.5s
700:	learn: 135291.5584000	test: 142472.8777143	best: 142472.8777143 (700)	total: 20.6s	remaining: 23.5s
800:	learn: 135290.6368000	test: 142471.9725714	best: 142471.9725714 (800)	total: 23.5s	remaining: 20.6s
900:	learn: 135289.7408000	test: 142471.0674286	best: 142471

,выборка,даты,строки,WAPE,|Relative Bias|,метрика конкурса,среднее по дням
0,train,40,40000,0.477117,0.092024,0.569141,0.581518
1,test,14,14000,0.480003,0.126466,0.606469,0.605725


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 136272.6528000	test: 141211.4468571	best: 141211.4468571 (0)	total: 27.6ms	remaining: 41.3s
100:	learn: 136271.8976000	test: 141210.6971429	best: 141210.6880000 (99)	total: 2.85s	remaining: 39.5s
200:	learn: 136271.1424000	test: 141209.9291429	best: 141209.9291429 (198)	total: 5.62s	remaining: 36.4s
300:	learn: 136270.3872000	test: 141209.1611429	best: 141209.1611429 (300)	total: 8.33s	remaining: 33.2s
400:	learn: 136269.6704000	test: 141208.4022857	best: 141208.4022857 (399)	total: 11.1s	remaining: 30.4s
500:	learn: 136268.8896000	test: 141207.6525714	best: 141207.6525714 (500)	total: 13.9s	remaining: 27.6s
600:	learn: 136268.1344000	test: 141206.8845714	best: 141206.8845714 (598)	total: 16.6s	remaining: 24.8s
700:	learn: 136267.3664000	test: 141206.1165714	best: 141206.1165714 (699)	total: 19.3s	remaining: 22s
800:	learn: 136266.6112000	test: 141205.3394286	best: 141205.3394286 (800)	total: 22.1s	remaining: 19.3s
900:	learn: 136265.8688000	test: 141204.5897143	best: 141204.

,выборка,даты,строки,WAPE,|Relative Bias|,метрика конкурса,среднее по дням
0,train,40,40000,0.474613,0.089155,0.563769,0.568768
1,test,14,14000,0.479977,0.112907,0.592884,0.592201


... (17 outputs left out)

,выборка,даты,строки,WAPE,|Relative Bias|,метрика конкурса,среднее по дням
0,train,40,40000,0.475922,0.090949,0.566871,0.577695
1,test,14,14000,0.480363,0.090195,0.570558,0.569053


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 137229.8368000	test: 139243.8125714	best: 139243.8125714 (0)	total: 28ms	remaining: 42s
100:	learn: 137228.7744000	test: 139242.7977143	best: 139242.7977143 (99)	total: 2.79s	remaining: 38.6s
200:	learn: 137227.7760000	test: 139241.7920000	best: 139241.7920000 (200)	total: 5.51s	remaining: 35.6s
300:	learn: 137226.7776000	test: 139240.7954286	best: 139240.7954286 (300)	total: 8.21s	remaining: 32.7s
400:	learn: 137225.8048000	test: 139239.8171429	best: 139239.8171429 (400)	total: 10.9s	remaining: 29.9s
500:	learn: 137224.8192000	test: 139238.8388571	best: 139238.8388571 (499)	total: 13.6s	remaining: 27.2s
600:	learn: 137223.8080000	test: 139237.8514286	best: 139237.8514286 (599)	total: 16.3s	remaining: 24.4s
700:	learn: 137222.7840000	test: 139236.8457143	best: 139236.8457143 (698)	total: 19s	remaining: 21.6s
800:	learn: 137221.7856000	test: 139235.8400000	best: 139235.8400000 (800)	total: 21.7s	remaining: 18.9s
900:	learn: 137220.7744000	test: 139234.8251429	best: 139234.8251

,выборка,даты,строки,WAPE,|Relative Bias|,метрика конкурса,среднее по дням
0,train,40,40000,0.472835,0.090894,0.563729,0.578430
1,test,14,14000,0.480485,0.098580,0.579065,0.577523


## Смешивание и калибровка
Подбираем общий вес модели и множитель на ранних проверочных датах. Сравниваем исходные и скорректированные прогнозы на последних 14 датах. Для submission используем настройки раннего периода. Оценка каждой даты объединяет все 8 горизонтов и все маршруты.


In [9]:
def blend(frame, weight):
    return weight * frame.catboost + (1 - weight) * frame.seasonal

tuning = validation.loc[validation.cutoff < holdout_start]
holdout = validation.loc[validation.cutoff >= holdout_start]
candidates = []
for w in np.linspace(0, 1, 21):
    mixed = blend(tuning, w)
    for k in (1.0, tuning.y.sum() / mixed.sum()):
        candidates.append((mean_daily_score(tuning, mixed * k), w, k))
_, weight, factor = min(candidates)
print(f"Вес CatBoost: {weight:.2f}, множитель: {factor:.4f}")
for label, prediction in [("Сезонный", holdout.seasonal), ("CatBoost", holdout.catboost),
                          ("Смесь", blend(holdout, weight)),
                          ("Смесь с калибровкой", blend(holdout, weight) * factor)]:
    print(f"{label}: {mean_daily_score(holdout, prediction):.5f}")

Вес CatBoost: 0.10, множитель: 0.9955
Сезонный: 0.38169
CatBoost: 0.57153
Смесь: 0.37416
Смесь с калибровкой: 0.37266


## Три эксперимента
Сравниваем две модели h=2 в исходных единицах. Для первого горизонта проверяем замену всей исходной смеси прогнозом из перекрытия. Для остальных пробуем добавить нормализованный профиль слота за 28 дней. Вес профиля и состав итогового прогноза выбираем по первым 14 проверочным датам, а последние 14 дней и субботы показываем отдельно. В первый день истории признаков для профиля доступны 27 полных прошлых суточных окон.


In [10]:
def experiment_prediction(frame, overlap, scaled, profile_weight):
    base = blend(frame, weight) * factor
    if scaled:
        second = frame.horizon.eq(2)
        base.loc[second] = (weight * frame.loc[second, "scaled"] +
                            (1 - weight) * frame.loc[second, "seasonal"]) * factor
    if overlap:
        first = frame.horizon.eq(1)
        base.loc[first] = frame.loc[first, "overlap"]
    later = frame.horizon.gt(1)
    base.loc[later] = (1 - profile_weight) * base.loc[later] + profile_weight * frame.loc[later, "profile28"]
    return base

second = holdout.loc[holdout.horizon.eq(2)]
display(pd.DataFrame([{"модель": name, "MAE": (second[name] - second.y).abs().mean(),
                       "std прогноза": second[name].std(), "метрика h2": mean_daily_score(second, second[name])}
                      for name in ("catboost", "scaled")]))
choices = [(overlap, scaled, float(w)) for overlap in (False, True)
           for scaled in (False, True) for w in np.linspace(0, 1, 11)]
def tune_score(choice):
    return mean_daily_score(tuning, experiment_prediction(tuning, *choice))

chosen = min(choices, key=tune_score)
profile_only = min([c for c in choices if not c[0] and not c[1]], key=tune_score)
rows = []
for label, choice in [("исходная смесь", (False, False, 0)), ("только замена h1", (True, False, 0)),
                      ("только замена модели h2", (False, True, 0)), ("только профиль", profile_only),
                      ("выбранная комбинация", chosen)]:
    row = {"вариант": label, "вес профиля": choice[2], "подбор": tune_score(choice)}
    for period, subset in [("проверка", holdout), ("субботы", holdout.loc[holdout.cutoff.dt.dayofweek.eq(5)])]:
        row[period] = mean_daily_score(subset, experiment_prediction(subset, *choice))
    rows.append(row)
display(pd.DataFrame(rows))
print("Замена h1, масштабирование h2, вес профиля:", chosen)


,модель,MAE,std прогноза,метрика h2
0,catboost,141488.197025,22.026455,0.592201
1,scaled,92784.307398,122999.340180,0.397888


,вариант,вес профиля,подбор,проверка,субботы
0,исходная смесь,0.0,0.378195,0.372661,0.361657
1,только замена h1,0.0,0.367483,0.362204,0.352044
2,только замена модели h2,0.0,0.377248,0.371892,0.360755
3,только профиль,0.6,0.367912,0.364753,0.364048
4,выбранная комбинация,0.6,0.357468,0.355358,0.355689


Замена h1, масштабирование h2, вес профиля: (True, True, 0.6000000000000001)


In [11]:
early = pd.read_parquet(OUT / "history_h1.parquet")
early = early.loc[early.cutoff < tune_start].copy()
early["seasonal"] = early[["slot_daily7", "slot_weekly4"]].mean(axis=1)
print("Ранняя история, метрика отдельно для h1:")
for name in ("seasonal", "overlap"):
    print(name, mean_daily_score(early, early[name]))
del early

Ранняя история, метрика отдельно для h1:
seasonal 0.4164050327678953
overlap 0.3284227756247388


## Финальное обучение на всей размеченной истории
После выбора настроек переобучаем все варианты с сохранённым числом деревьев. После каждого обучения показываем оценку на всей истории и сохранённые train/test предыдущей модели. Даты локального test теперь включены в обучение, поэтому независимая оценка относится к предыдущему этапу.


In [12]:
submission_parts = []
for h in range(1, 9):
    data = pd.read_parquet(OUT / f"history_h{h}.parquet")
    future = pd.read_parquet(OUT / f"submission_h{h}.parquet")
    columns = model_columns(data)
    result = future[["id", "profile28", "overlap"]].copy()
    result["seasonal"] = future[["slot_daily7", "slot_weekly4"]].mean(axis=1)
    result["horizon"] = h
    for variant, scale in variants(h):
        model = make_model(tree_counts[h, variant])
        model.fit(data[columns], data.y / scale)
        suffix = "_scaled" if variant == "scaled" else ""
        model.save_model(str(MODEL_DIR / f"catboost_h{h}{suffix}.cbm"))
        result[variant] = (model.predict(future[columns]) * scale).clip(min=0)
        full_pred = (model.predict(data[columns]) * scale).clip(min=0)
        full_row = score_row(data, full_pred, "вся история после переобучения")
        print(f"h={h}, {variant}: первые две строки — оценка до переобучения")
        display(pd.concat([evaluation_tables[h, variant], pd.DataFrame([full_row])], ignore_index=True))
        del model
    submission_parts.append(result)

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 138029.2668235	total: 13.7ms	remaining: 20.5s
100:	learn: 138028.3482353	total: 1.2s	remaining: 16.6s
200:	learn: 138027.4597647	total: 2.39s	remaining: 15.4s
300:	learn: 138026.5562353	total: 3.57s	remaining: 14.2s
400:	learn: 138025.6527059	total: 4.76s	remaining: 13s
500:	learn: 138024.7642353	total: 5.95s	remaining: 11.9s
600:	learn: 138023.8757647	total: 7.14s	remaining: 10.7s
700:	learn: 138022.9872941	total: 8.33s	remaining: 9.49s
800:	learn: 138022.0536471	total: 9.5s	remaining: 8.29s
900:	learn: 138021.1651765	total: 10.7s	remaining: 7.11s
1000:	learn: 138020.2616471	total: 11.8s	remaining: 5.9s
1100:	learn: 138019.3129412	total: 13s	remaining: 4.71s
1200:	learn: 138018.4395294	total: 14.1s	remaining: 3.52s
1300:	learn: 138017.5360000	total: 15.3s	remaining: 2.34s
1400:	learn: 138016.6475294	total: 16.4s	remaining: 1.16s
1499:	learn: 138015.7440000	total: 17.6s	remaining: 0us
h=1, catboost: первые две строки — оценка до переобучения


,выборка,даты,строки,WAPE,|Relative Bias|,метрика конкурса,среднее по дням
0,train,40,40000,0.477117,0.092024,0.569141,0.581518
1,test,14,14000,0.480003,0.126466,0.606469,0.605725
2,вся история после переобучения,68,68000,0.478950,0.096232,0.575182,0.582819


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 138347.1887059	total: 12ms	remaining: 18s
100:	learn: 138346.0141176	total: 1.16s	remaining: 16s
200:	learn: 138344.8545882	total: 2.31s	remaining: 14.9s
300:	learn: 138343.7251765	total: 3.45s	remaining: 13.7s
400:	learn: 138342.5505882	total: 4.59s	remaining: 12.6s
500:	learn: 138341.4211765	total: 5.73s	remaining: 11.4s
600:	learn: 138340.2616471	total: 6.85s	remaining: 10.2s
700:	learn: 138339.0870588	total: 7.98s	remaining: 9.07s
800:	learn: 138337.8974118	total: 9.14s	remaining: 7.95s
900:	learn: 138336.7228235	total: 10.3s	remaining: 6.84s
1000:	learn: 138335.5632941	total: 11.5s	remaining: 5.7s
1100:	learn: 138334.3736471	total: 12.6s	remaining: 4.55s
1200:	learn: 138333.2743529	total: 13.8s	remaining: 3.4s
1300:	learn: 138332.0696471	total: 14.9s	remaining: 2.26s
1400:	learn: 138330.9101176	total: 16.1s	remaining: 1.11s
1497:	learn: 138329.7807059	total: 17.2s	remaining: 0us
h=2, catboost: первые две строки — оценка до переобучения


,выборка,даты,строки,WAPE,|Relative Bias|,метрика конкурса,среднее по дням
0,train,40,40000,0.474613,0.089155,0.563769,0.568768
1,test,14,14000,0.479977,0.112907,0.592884,0.592201
2,вся история после переобучения,68,68000,0.475786,0.090722,0.566508,0.570245


... (17 outputs left out)

,выборка,даты,строки,WAPE,|Relative Bias|,метрика конкурса,среднее по дням
0,train,40,40000,0.475922,0.090949,0.566871,0.577695
1,test,14,14000,0.480363,0.090195,0.570558,0.569053
2,вся история после переобучения,68,68000,0.477505,0.090968,0.568472,0.574637


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 138344.6437647	total: 12.3ms	remaining: 18.5s
100:	learn: 138343.6498824	total: 1.17s	remaining: 16.2s
200:	learn: 138342.6108235	total: 2.33s	remaining: 15.1s
300:	learn: 138341.6169412	total: 3.47s	remaining: 13.8s
400:	learn: 138340.6080000	total: 4.62s	remaining: 12.7s
500:	learn: 138339.5990588	total: 5.79s	remaining: 11.5s
600:	learn: 138338.5600000	total: 6.95s	remaining: 10.4s
700:	learn: 138337.5661176	total: 8.1s	remaining: 9.23s
800:	learn: 138336.5872941	total: 9.24s	remaining: 8.06s
900:	learn: 138335.5783529	total: 10.4s	remaining: 6.91s
1000:	learn: 138334.5995294	total: 11.5s	remaining: 5.76s
1100:	learn: 138333.5604706	total: 12.7s	remaining: 4.6s
1200:	learn: 138332.5063529	total: 13.9s	remaining: 3.45s
1300:	learn: 138331.5425882	total: 15.1s	remaining: 2.3s
1400:	learn: 138330.5336471	total: 16.2s	remaining: 1.15s
1499:	learn: 138329.5247059	total: 17.4s	remaining: 0us
h=8, catboost: первые две строки — оценка до переобучения


,выборка,даты,строки,WAPE,|Relative Bias|,метрика конкурса,среднее по дням
0,train,40,40000,0.472835,0.090894,0.563729,0.578430
1,test,14,14000,0.480485,0.098580,0.579065,0.577523
2,вся история после переобучения,68,68000,0.474838,0.091370,0.566208,0.574876


## Итоговый файл для соревнования


In [13]:
predictions = pd.concat(submission_parts, ignore_index=True)
predictions["y_pred"] = experiment_prediction(predictions, *chosen)
submission = test[["id"]].merge(predictions[["id", "y_pred"]], on="id", how="left",
                                validate="one_to_one")
if len(predictions) != len(test) or not np.isfinite(submission.y_pred).all():
    raise ValueError("Неполный прогноз для тестовой выборки")
submission.to_csv("submission.csv", index=False)
display(submission.head())

,id,y_pred
0,0,100030.758929
1,1,171556.558033
2,2,165805.167085
3,3,138609.411971
4,4,150256.175669
